In [1]:
# vibed using claude
import optuna
import xgboost as xgb
import lightgbm as lgb
import numpy as np
from sklearn.metrics import root_mean_squared_error


def tune_xgb(df_train, df_test, features, n_trials=100):
    X_train = df_train[features]
    y_train = df_train['target']
    X_test  = df_test[features]
    y_test  = df_test['target']

    def objective(trial):
        params = {
            "device":             "cuda",
            "objective":          "reg:squarederror",
            "n_estimators":       5000,
            "learning_rate":      trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
            "max_depth":          trial.suggest_int("max_depth", 4, 10),
            "subsample":          trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree":   trial.suggest_float("colsample_bytree", 0.4, 1.0),
            "colsample_bylevel":  trial.suggest_float("colsample_bylevel", 0.4, 1.0),
            "min_child_weight":   trial.suggest_int("min_child_weight", 1, 50),
            "reg_alpha":          trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "reg_lambda":         trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
            "gamma":              trial.suggest_float("gamma", 0.0, 5.0),
            "early_stopping_rounds": 100,
            "random_state":       42,
        }

        model = xgb.XGBRegressor(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            verbose=False
        )

        preds = model.predict(X_test)
        return root_mean_squared_error(y_test, preds)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    print("\n=== XGB Best trial ===")
    print(f"  RMSE : {study.best_value:.4f}")
    print(f"  Params: {study.best_params}")

    # Retrain with best params on full data
    best = study.best_params
    best_model = xgb.XGBRegressor(
        device="cuda",
        objective="reg:squarederror",
        n_estimators=5000,
        early_stopping_rounds=100,
        random_state=42,
        **best
    )
    best_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=100
    )

    return best_model, study


def tune_lgbm(df_train, df_test, features, n_trials=100):
    X_train = df_train[features]
    y_train = df_train['target']
    X_test  = df_test[features]
    y_test  = df_test['target']

    def objective(trial):
        params = {
            "n_estimators":       3000,
            "learning_rate":      trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
            "num_leaves":         trial.suggest_int("num_leaves", 16, 256),
            "max_depth":          trial.suggest_int("max_depth", 3, 12),
            "subsample":          trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree":   trial.suggest_float("colsample_bytree", 0.4, 1.0),
            "min_child_samples":  trial.suggest_int("min_child_samples", 5, 100),
            "reg_alpha":          trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "reg_lambda":         trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
            "subsample_freq":     trial.suggest_int("subsample_freq", 1, 10),
            "random_state":       42,
        }

        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            eval_metric="rmse",
            callbacks=[
                lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(period=-1)   # silences per-iter output
            ]
        )

        preds = model.predict(X_test)
        return root_mean_squared_error(y_test, preds)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    print("\n=== LGBM Best trial ===")
    print(f"  RMSE : {study.best_value:.4f}")
    print(f"  Params: {study.best_params}")

    # Retrain with best params
    best = study.best_params
    best_model = lgb.LGBMRegressor(
        n_estimators=3000,
        random_state=42,
        **best
    )
    best_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        eval_metric="rmse",
        callbacks=[
            lgb.early_stopping(50),
            lgb.log_evaluation(period=100)
        ]
    )

    return best_model, study



/home/matt/repos/nitor-comp/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from dataengineers import Dataset

ds = Dataset('train')
train, test = ds.build_train_test()
features = [c for c in train.columns if c not in ['target', 'id', 'delivery_start']]

In [4]:
xgb_model, xgb_study = tune_xgb(train, test, features, n_trials=100)

[I 2026-02-22 13:00:52,013] A new study created in memory with name: no-name-3e3c00cf-fcad-4fc4-8b8e-09a3526583af
  0%|          | 0/100 [00:00<?, ?it/s]/home/matt/repos/nitor-comp/.venv/lib/python3.14/site-packages/xgboost/core.py:751: UserWarning: [13:00:52] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
Best trial: 0. Best value: 52.3616:   1%|          | 1/100 [00:00<01:37,  1.01it/s]

[I 2026-02-22 13:00:52,999] Trial 0 finished with value: 52.3615845968335 and parameters: {'learning_rate': 0.027615686469877814, 'max_depth': 5, 'subsample': 0.6489739002785404, 'colsample_bytree': 0.5720585152100733, 'colsample_bylevel': 0.7306083778215492, 'min_child_weight': 22, 'reg_alpha': 1.103667132161936, 'reg_lambda': 0.002896160408594056, 'gamma': 2.837904317061435}. Best is trial 0 with value: 52.3615845968335.


Best trial: 0. Best value: 52.3616:   2%|▏         | 2/100 [00:01<01:31,  1.08it/s]

[I 2026-02-22 13:00:53,888] Trial 1 finished with value: 55.312088916176265 and parameters: {'learning_rate': 0.008147828476956508, 'max_depth': 4, 'subsample': 0.6049161014528887, 'colsample_bytree': 0.538047486850476, 'colsample_bylevel': 0.9504407341143403, 'min_child_weight': 2, 'reg_alpha': 1.1800673561807757, 'reg_lambda': 0.0006200616746481541, 'gamma': 4.716158761416258}. Best is trial 0 with value: 52.3615845968335.


Best trial: 0. Best value: 52.3616:   3%|▎         | 3/100 [00:02<01:23,  1.16it/s]

...

[I 2026-02-22 13:04:11,397] Trial 97 finished with value: 41.9178459580093 and parameters: {'learning_rate': 0.04637003804677165, 'max_depth': 10, 'subsample': 0.6678075801288558, 'colsample_bytree': 0.4370173998696182, 'colsample_bylevel': 0.6493429881926125, 'min_child_weight': 1, 'reg_alpha': 0.7599158808621777, 'reg_lambda': 7.430805387038759, 'gamma': 0.7773657979184866}. Best is trial 82 with value: 41.71686680276176.


Best trial: 98. Best value: 41.258:  99%|█████████▉| 99/100 [03:21<00:02,  2.11s/it] 

[I 2026-02-22 13:04:13,761] Trial 98 finished with value: 41.258033055276776 and parameters: {'learning_rate': 0.04646125525152466, 'max_depth': 10, 'subsample': 0.6897202767737011, 'colsample_bytree': 0.40179230169157854, 'colsample_bylevel': 0.6517806761012462, 'min_child_weight': 1, 'reg_alpha': 0.7080207458963536, 'reg_lambda': 9.729317474100121, 'gamma': 0.8003392666357263}. Best is trial 98 with value: 41.258033055276776.


Best trial: 98. Best value: 41.258: 100%|██████████| 100/100 [03:23<00:00,  2.03s/it]


[I 2026-02-22 13:04:15,369] Trial 99 finished with value: 47.39490710974027 and parameters: {'learning_rate': 0.04676040839139747, 'max_depth': 9, 'subsample': 0.6566526170049661, 'colsample_bytree': 0.40048466567853325, 'colsample_bylevel': 0.5621137553190889, 'min_child_weight': 26, 'reg_alpha': 0.7309532698758907, 'reg_lambda': 8.131271118151455, 'gamma': 0.7845809261477318}. Best is trial 98 with value: 41.258033055276776.

=== XGB Best trial ===
  RMSE : 41.2580
  Params: {'learning_rate': 0.04646125525152466, 'max_depth': 10, 'subsample': 0.6897202767737011, 'colsample_bytree': 0.40179230169157854, 'colsample_bylevel': 0.6517806761012462, 'min_child_weight': 1, 'reg_alpha': 0.7080207458963536, 'reg_lambda': 9.729317474100121, 'gamma': 0.8003392666357263}
[0]	validation_0-rmse:62.27494
[100]	validation_0-rmse:42.29947
[183]	validation_0-rmse:46.19656


In [ ]:
lgb_model, lgb_study  = tune_lgbm(train, test, features, n_trials=100)